# Segmentation

A second reduction lever, working *inside* each period. Adjacent timesteps that look alike are
merged into one **segment** with one value, so a 24-hour day might be described by 6 segments.
Segments are **variable length** — long where the profile is flat, short where it moves.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data
from tsam import SegmentConfig

week = slice("2010-01-11", "2010-01-17")

## 24 hours → 6 segments

The reconstruction is a step function: each flat step is one segment, held constant across the
hours it covers.

In [ ]:
result = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    segments=SegmentConfig(n_segments=6),
)
result.plot.compare(columns=["Load"], time_slice=week, color="source")

## What each segment keeps

Every segment carries the **mean** of the hours it covers by default. That is the only thing you
tune about how a segment is described — the merging algorithm itself is fixed (constrained
agglomerative clustering, adjacent steps only), so there is no segmentation "method", only
`n_segments` and `representation`. The choices are the same as for
[period representations](representations.ipynb):

In [ ]:
result_medoid = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    segments=SegmentConfig(n_segments=6, representation="medoid"),
)
result_medoid.plot.compare(columns=["Load"], time_slice=week, color="source")

## What it buys

The two levers multiply:

In [ ]:
plain = tsam.aggregate(data, n_clusters=6, period_duration="1D")

original_steps = len(data)
plain_steps = plain.n_clusters * plain.n_timesteps_per_period
seg_steps = result.n_clusters * result.n_segments

pd.DataFrame(
    {
        "timesteps": [original_steps, plain_steps, seg_steps],
        "reduction": [
            "—",
            f"{1 - plain_steps / original_steps:.0%}",
            f"{1 - seg_steps / original_steps:.0%}",
        ],
        "mean RMSE": [
            0.0,
            round(float(plain.accuracy.rmse.mean()), 4),
            round(float(result.accuracy.rmse.mean()), 4),
        ],
    },
    index=["original (hourly)", "6 days x 24 h", "6 days x 6 segments"],
)

Segments are variable-length and differ between typical periods, so every downstream sum must
weight by `result.segment_durations`.

---

* [How small can you go?](tuning.ipynb) — search periods and segments together for a target size.
* [Choosing a method](../tutorials/choosing_a_method.ipynb) — when to spend on segments rather
  than periods.
* [Segmentation](../explanation/how-aggregation-works/06_segmentation.ipynb) — the merging
  algorithm.